# 03 — Build Payment Recovery Gold Model

## Purpose

Build an invoice-level payment recovery model that analyzes payment journeys from the first collection attempt through retry resolution.

The model measures first-attempt success, retry recovery, unresolved failures, pending collections, recovery duration, recovered revenue, and payment-provider performance.

## Grain

One record per invoice with at least one validated Silver payment attempt.

## Sources

- `workspace.revenue_leakage_silver.payments`
- `workspace.revenue_leakage_gold.revenue_leakage`

## Target

- `workspace.revenue_leakage_gold.payment_recovery`

## Business Outcomes

- Measure first-attempt payment success
- Quantify successful and failed retry journeys
- Identify unresolved payment failures
- Calculate recovered retry revenue
- Compare payment providers and payment methods
- Prioritize recovery opportunities by customer risk and value

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SILVER_PAYMENTS_TABLE = (
    "workspace.revenue_leakage_silver.payments"
)

GOLD_REVENUE_LEAKAGE_TABLE = (
    "workspace.revenue_leakage_gold.revenue_leakage"
)

GOLD_PAYMENT_RECOVERY_TABLE = (
    "workspace.revenue_leakage_gold.payment_recovery"
)

EXPECTED_PAYMENT_COUNT = 29_972
EXPECTED_PAYMENT_JOURNEY_COUNT = 25_748
EXPECTED_REVENUE_LEAKAGE_COUNT = 26_699


silver_payments_df = spark.table(
    SILVER_PAYMENTS_TABLE
)

revenue_leakage_source_df = spark.table(
    GOLD_REVENUE_LEAKAGE_TABLE
)


payment_count = silver_payments_df.count()

distinct_payment_count = (
    silver_payments_df
    .select("payment_id")
    .distinct()
    .count()
)

payment_journey_count = (
    silver_payments_df
    .select("invoice_id")
    .distinct()
    .count()
)

revenue_leakage_count = (
    revenue_leakage_source_df.count()
)

distinct_revenue_leakage_invoice_count = (
    revenue_leakage_source_df
    .select("invoice_id")
    .distinct()
    .count()
)

payment_invoice_reference_errors = (
    silver_payments_df
    .select(
        "payment_id",
        "invoice_id",
    )
    .join(
        revenue_leakage_source_df
        .select("invoice_id"),
        on="invoice_id",
        how="left_anti",
    )
    .count()
)


assert payment_count == EXPECTED_PAYMENT_COUNT, (
    "Unexpected Silver payment count."
)

assert distinct_payment_count == EXPECTED_PAYMENT_COUNT, (
    "Duplicate Silver payment IDs detected."
)

assert (
    payment_journey_count
    == EXPECTED_PAYMENT_JOURNEY_COUNT
), (
    "Unexpected distinct payment journey count."
)

assert (
    revenue_leakage_count
    == EXPECTED_REVENUE_LEAKAGE_COUNT
), (
    "Unexpected Revenue Leakage source count."
)

assert (
    distinct_revenue_leakage_invoice_count
    == EXPECTED_REVENUE_LEAKAGE_COUNT
), (
    "Duplicate Revenue Leakage invoice IDs detected."
)

assert payment_invoice_reference_errors == 0, (
    "Payments reference missing Revenue Leakage invoices."
)


print(
    "Silver payment attempts: "
    f"{payment_count:,}"
)

print(
    "Distinct payment IDs: "
    f"{distinct_payment_count:,}"
)

print(
    "Distinct invoice payment journeys: "
    f"{payment_journey_count:,}"
)

print(
    "Revenue Leakage source records: "
    f"{revenue_leakage_count:,}"
)

print(
    "Payment/invoice reference errors: "
    f"{payment_invoice_reference_errors:,}"
)

print(
    "All required Payment Recovery sources are available."
)

display(
    silver_payments_df
    .groupBy(
        "payment_status",
        "attempt_number",
    )
    .agg(
        F.count("*").alias(
            "payment_count"
        ),
        F.round(
            F.sum("transaction_amount"),
            2,
        ).alias(
            "transaction_amount"
        ),
        F.round(
            F.sum("settled_amount"),
            2,
        ).alias(
            "settled_amount"
        ),
    )
    .orderBy(
        "payment_status",
        "attempt_number",
    )
)

## 2. Build Invoice Payment Journeys

Sequence every payment attempt within its invoice and retain the first attempt, latest attempt, failure history, retry outcome, recovery date, and financial totals.

Each invoice payment journey represents the complete collection lifecycle for one invoice.

In [0]:
first_payment_attempt_window = (
    Window
    .partitionBy("invoice_id")
    .orderBy(
        F.col("attempt_number").asc(),
        F.col("payment_date").asc_nulls_last(),
        F.col("payment_id").asc(),
    )
)

latest_payment_attempt_window = (
    Window
    .partitionBy("invoice_id")
    .orderBy(
        F.col("attempt_number").desc(),
        F.col("payment_date").desc_nulls_last(),
        F.col("payment_id").desc(),
    )
)


first_payment_attempt_df = (
    silver_payments_df
    .withColumn(
        "_first_attempt_rank",
        F.row_number().over(
            first_payment_attempt_window
        ),
    )
    .filter(
        F.col("_first_attempt_rank") == 1
    )
    .select(
        "invoice_id",
        F.col("payment_id").alias(
            "first_payment_id"
        ),
        F.col("provider_transaction_id").alias(
            "first_provider_transaction_id"
        ),
        F.col("payment_status").alias(
            "first_payment_status"
        ),
        F.col("attempt_number").alias(
            "first_attempt_number"
        ),
        F.col("payment_date").alias(
            "first_payment_date"
        ),
        F.col("payment_method").alias(
            "first_payment_method"
        ),
        F.col("payment_provider").alias(
            "first_payment_provider"
        ),
        F.col("failure_reason").alias(
            "first_failure_reason"
        ),
        F.col("transaction_amount").alias(
            "first_attempt_amount"
        ),
        F.col("settled_amount").alias(
            "first_settled_amount"
        ),
    )
)


latest_payment_attempt_df = (
    silver_payments_df
    .withColumn(
        "_latest_attempt_rank",
        F.row_number().over(
            latest_payment_attempt_window
        ),
    )
    .filter(
        F.col("_latest_attempt_rank") == 1
    )
    .select(
        "invoice_id",
        F.col("payment_id").alias(
            "latest_payment_id"
        ),
        F.col("provider_transaction_id").alias(
            "latest_provider_transaction_id"
        ),
        F.col("payment_status").alias(
            "latest_payment_status"
        ),
        F.col("attempt_number").alias(
            "latest_attempt_number"
        ),
        F.col("payment_date").alias(
            "latest_payment_date"
        ),
        F.col("settlement_date").alias(
            "latest_settlement_date"
        ),
        F.col("payment_method").alias(
            "latest_payment_method"
        ),
        F.col("payment_provider").alias(
            "latest_payment_provider"
        ),
        F.col("failure_reason").alias(
            "latest_failure_reason"
        ),
        F.col("transaction_amount").alias(
            "latest_attempt_amount"
        ),
        F.col("settled_amount").alias(
            "latest_settled_amount"
        ),
    )
)


payment_journey_aggregates_df = (
    silver_payments_df
    .groupBy("invoice_id")
    .agg(
        F.count("*").alias(
            "payment_attempt_count"
        ),
        F.countDistinct("payment_id").alias(
            "distinct_payment_count"
        ),
        F.countDistinct(
            "provider_transaction_id"
        ).alias(
            "distinct_provider_transaction_count"
        ),
        F.max("attempt_number").alias(
            "maximum_attempt_number"
        ),
        F.sum(
            F.when(
                F.col("payment_status")
                == "Succeeded",
                1,
            ).otherwise(0)
        ).alias(
            "successful_payment_count"
        ),
        F.sum(
            F.when(
                F.col("payment_status")
                == "Failed",
                1,
            ).otherwise(0)
        ).alias(
            "failed_payment_count"
        ),
        F.sum(
            F.when(
                F.col("payment_status")
                == "Pending",
                1,
            ).otherwise(0)
        ).alias(
            "pending_payment_count"
        ),
        F.sum(
            F.when(
                F.col("attempt_number") > 1,
                1,
            ).otherwise(0)
        ).alias(
            "retry_attempt_count"
        ),
        F.sum(
            F.when(
                (
                    F.col("attempt_number") > 1
                )
                & (
                    F.col("payment_status")
                    == "Succeeded"
                ),
                1,
            ).otherwise(0)
        ).alias(
            "successful_retry_count"
        ),
        F.sum(
            F.when(
                (
                    F.col("attempt_number") > 1
                )
                & (
                    F.col("payment_status")
                    == "Failed"
                ),
                1,
            ).otherwise(0)
        ).alias(
            "failed_retry_count"
        ),
        F.round(
            F.sum("transaction_amount"),
            2,
        ).alias(
            "payment_attempt_amount"
        ),
        F.round(
            F.sum("settled_amount"),
            2,
        ).alias(
            "settled_amount"
        ),
        F.round(
            F.sum(
                F.when(
                    F.col("payment_status")
                    == "Failed",
                    F.col("transaction_amount"),
                ).otherwise(0)
            ),
            2,
        ).alias(
            "failed_attempt_amount"
        ),
        F.round(
            F.sum(
                F.when(
                    F.col("payment_status")
                    == "Pending",
                    F.col("transaction_amount"),
                ).otherwise(0)
            ),
            2,
        ).alias(
            "pending_attempt_amount"
        ),
        F.round(
            F.sum(
                F.when(
                    (
                        F.col("attempt_number") > 1
                    )
                    & (
                        F.col("payment_status")
                        == "Succeeded"
                    ),
                    F.col("settled_amount"),
                ).otherwise(0)
            ),
            2,
        ).alias(
            "recovered_retry_amount"
        ),
        F.min(
            F.when(
                F.col("payment_status")
                == "Failed",
                F.col("payment_date"),
            )
        ).alias(
            "first_failure_date"
        ),
        F.min(
            F.when(
                F.col("payment_status")
                == "Succeeded",
                F.col("payment_date"),
            )
        ).alias(
            "first_success_date"
        ),
        F.min(
            F.when(
                (
                    F.col("attempt_number") > 1
                )
                & (
                    F.col("payment_status")
                    == "Succeeded"
                ),
                F.col("payment_date"),
            )
        ).alias(
            "recovery_date"
        ),
        F.array_sort(
            F.collect_set("failure_reason")
        ).alias(
            "failure_reasons"
        ),
        F.array_sort(
            F.collect_set("payment_method")
        ).alias(
            "payment_methods"
        ),
        F.array_sort(
            F.collect_set("payment_provider")
        ).alias(
            "payment_providers"
        ),
    )
)


payment_journey_df = (
    payment_journey_aggregates_df
    .join(
        first_payment_attempt_df,
        on="invoice_id",
        how="inner",
    )
    .join(
        latest_payment_attempt_df,
        on="invoice_id",
        how="inner",
    )
    .withColumn(
        "recovery_duration_days",
        F.when(
            F.col("recovery_date").isNotNull(),
            F.datediff(
                F.col("recovery_date"),
                F.col("first_failure_date"),
            ),
        ),
    )
)


payment_journey_record_count = (
    payment_journey_df.count()
)

distinct_payment_journey_count = (
    payment_journey_df
    .select("invoice_id")
    .distinct()
    .count()
)

duplicate_payment_journey_count = (
    payment_journey_record_count
    - distinct_payment_journey_count
)

payment_journey_totals = (
    payment_journey_df
    .agg(
        F.sum("payment_attempt_count").alias(
            "payment_attempt_count"
        ),
        F.round(
            F.sum("payment_attempt_amount"),
            2,
        ).alias(
            "payment_attempt_amount"
        ),
        F.round(
            F.sum("settled_amount"),
            2,
        ).alias(
            "settled_amount"
        ),
        F.round(
            F.sum("recovered_retry_amount"),
            2,
        ).alias(
            "recovered_retry_amount"
        ),
    )
    .first()
)


assert (
    payment_journey_record_count
    == EXPECTED_PAYMENT_JOURNEY_COUNT
), (
    "Unexpected payment journey record count."
)

assert duplicate_payment_journey_count == 0, (
    "Duplicate invoice payment journeys detected."
)

assert (
    payment_journey_totals[
        "payment_attempt_count"
    ]
    == EXPECTED_PAYMENT_COUNT
), (
    "Payment journey attempts do not reconcile."
)


print(
    "Payment journey records: "
    f"{payment_journey_record_count:,}"
)

print(
    "Distinct invoice journeys: "
    f"{distinct_payment_journey_count:,}"
)

print(
    "Duplicate invoice journeys: "
    f"{duplicate_payment_journey_count:,}"
)

print(
    "Reconciled payment attempts: "
    f"{payment_journey_totals['payment_attempt_count']:,}"
)

print(
    "Payment attempt amount: "
    f"{payment_journey_totals['payment_attempt_amount']:,.2f}"
)

print(
    "Settled amount: "
    f"{payment_journey_totals['settled_amount']:,.2f}"
)

print(
    "Recovered retry amount: "
    f"{payment_journey_totals['recovered_retry_amount']:,.2f}"
)

display(
    payment_journey_df
    .groupBy(
        "first_payment_status",
        "latest_payment_status",
        "maximum_attempt_number",
    )
    .agg(
        F.count("*").alias(
            "journey_count"
        ),
        F.round(
            F.sum("settled_amount"),
            2,
        ).alias(
            "settled_amount"
        ),
        F.round(
            F.sum("recovered_retry_amount"),
            2,
        ).alias(
            "recovered_retry_amount"
        ),
    )
    .orderBy(
        "first_payment_status",
        "latest_payment_status",
        "maximum_attempt_number",
    )
)

## 3. Assemble the Payment Recovery Dataset

Enrich every invoice payment journey with invoice exposure, customer risk, customer value, and collection context.

Classify first-attempt success, retry recovery, failed retry, missing retry, and pending collection outcomes. Assign a recovery action and priority for unresolved payment journeys.

In [0]:
payment_recovery_context_df = (
    revenue_leakage_source_df
    .select(
        "invoice_id",
        "customer_id",
        "subscription_id",
        "customer_name",
        "customer_status",
        "customer_segment",
        "country",
        "region",
        "risk_score",
        "risk_tier",
        "customer_value_tier",
        "invoice_status",
        "invoice_total_amount",
        "amount_paid",
        "outstanding_amount",
        "due_date",
        "days_past_due",
        "confirmed_leakage_amount",
        "revenue_at_risk_amount",
        "leakage_status",
        "collection_priority",
        "analytics_snapshot_date",
    )
)


payment_recovery_joined_df = (
    payment_journey_df
    .join(
        payment_recovery_context_df,
        on="invoice_id",
        how="inner",
    )
)


payment_recovery_classified_df = (
    payment_recovery_joined_df
    .withColumn(
        "recovery_outcome",
        F.when(
            F.col("first_payment_status")
            == "Succeeded",
            "First Attempt Success",
        )
        .when(
            (
                F.col("first_payment_status")
                == "Failed"
            )
            & (
                F.col("latest_payment_status")
                == "Succeeded"
            )
            & (
                F.col("latest_attempt_number") > 1
            ),
            "Recovered by Retry",
        )
        .when(
            (
                F.col("first_payment_status")
                == "Failed"
            )
            & (
                F.col("latest_payment_status")
                == "Failed"
            )
            & (
                F.col("latest_attempt_number") > 1
            ),
            "Retry Failed",
        )
        .when(
            (
                F.col("first_payment_status")
                == "Failed"
            )
            & (
                F.col("latest_payment_status")
                == "Failed"
            )
            & (
                F.col("latest_attempt_number") == 1
            ),
            "No Retry Attempted",
        )
        .when(
            F.col("latest_payment_status")
            == "Pending",
            "Pending Collection",
        )
        .otherwise("Other"),
    )
    .withColumn(
        "recovery_status",
        F.when(
            F.col("first_payment_status")
            == "Succeeded",
            "Resolved",
        )
        .when(
            (
                F.col("first_payment_status")
                == "Failed"
            )
            & (
                F.col("latest_payment_status")
                == "Succeeded"
            ),
            "Recovered",
        )
        .when(
            F.col("latest_payment_status")
            == "Failed",
            "Unrecovered",
        )
        .when(
            F.col("latest_payment_status")
            == "Pending",
            "Pending",
        )
        .otherwise("Other"),
    )
    .withColumn(
        "first_attempt_success_flag",
        F.when(
            F.col("first_payment_status")
            == "Succeeded",
            1,
        ).otherwise(0),
    )
    .withColumn(
        "recovery_eligible_flag",
        F.when(
            F.col("first_payment_status")
            == "Failed",
            1,
        ).otherwise(0),
    )
    .withColumn(
        "recovery_success_flag",
        F.when(
            (
                F.col("first_payment_status")
                == "Failed"
            )
            & (
                F.col("latest_payment_status")
                == "Succeeded"
            ),
            1,
        ).otherwise(0),
    )
    .withColumn(
        "unrecovered_failure_flag",
        F.when(
            F.col("latest_payment_status")
            == "Failed",
            1,
        ).otherwise(0),
    )
    .withColumn(
        "pending_collection_flag",
        F.when(
            F.col("latest_payment_status")
            == "Pending",
            1,
        ).otherwise(0),
    )
    .withColumn(
        "recoverable_amount",
        F.when(
            F.col("first_payment_status")
            == "Failed",
            F.col("first_attempt_amount"),
        ).otherwise(0),
    )
    .withColumn(
        "recovered_amount",
        F.round(
            F.col("recovered_retry_amount"),
            2,
        ),
    )
    .withColumn(
        "unrecovered_amount",
        F.when(
            F.col("latest_payment_status")
            == "Failed",
            F.col("outstanding_amount"),
        ).otherwise(0),
    )
    .withColumn(
        "pending_collection_amount",
        F.when(
            F.col("latest_payment_status")
            == "Pending",
            F.col("pending_attempt_amount"),
        ).otherwise(0),
    )
)


raw_recovery_priority_score = (
    F.when(
        F.col("recovery_status")
        == "Unrecovered",
        40,
    ).otherwise(0)
    +
    F.when(
        F.col("recovery_status")
        == "Pending",
        25,
    ).otherwise(0)
    +
    F.when(
        F.col("outstanding_amount") > 0,
        20,
    ).otherwise(0)
    +
    F.when(
        (
            F.col("latest_payment_status")
            == "Failed"
        )
        & (
            F.col("latest_attempt_number") > 1
        ),
        15,
    ).otherwise(0)
    +
    F.when(
        F.col("days_past_due") > 90,
        20,
    )
    .when(
        F.col("days_past_due") > 60,
        15,
    )
    .when(
        F.col("days_past_due") > 30,
        10,
    )
    .when(
        F.col("days_past_due") > 0,
        5,
    )
    .otherwise(0)
    +
    F.when(
        F.col("risk_tier") == "High",
        15,
    )
    .when(
        F.col("risk_tier") == "Medium",
        8,
    )
    .otherwise(0)
    +
    F.when(
        F.col("customer_value_tier")
        == "High Value",
        10,
    ).otherwise(0)
)


payment_recovery_df = (
    payment_recovery_classified_df
    .withColumn(
        "recovery_priority_score",
        F.when(
            F.col("recovery_status").isin(
                "Unrecovered",
                "Pending",
            ),
            F.least(
                F.lit(100),
                raw_recovery_priority_score,
            ),
        ).otherwise(0),
    )
    .withColumn(
        "recovery_priority",
        F.when(
            F.col("recovery_priority_score")
            >= 75,
            "Critical",
        )
        .when(
            F.col("recovery_priority_score")
            >= 50,
            "High",
        )
        .when(
            F.col("recovery_priority_score")
            >= 25,
            "Medium",
        )
        .when(
            F.col("recovery_priority_score")
            > 0,
            "Low",
        )
        .otherwise("None"),
    )
    .withColumn(
        "recommended_recovery_action",
        F.when(
            F.col("recovery_outcome")
            == "Retry Failed",
            "Escalate Collection",
        )
        .when(
            F.col("recovery_outcome")
            == "No Retry Attempted",
            "Initiate Retry",
        )
        .when(
            F.col("recovery_outcome")
            == "Pending Collection",
            "Monitor Pending Payment",
        )
        .when(
            F.col("recovery_outcome")
            == "Recovered by Retry",
            "Monitor Recovered Customer",
        )
        .when(
            F.col("recovery_outcome")
            == "First Attempt Success",
            "No Action Required",
        )
        .otherwise("Review Payment Journey"),
    )
    .withColumn(
        "_gold_generated_at",
        F.current_timestamp(),
    )
)


payment_recovery_count = (
    payment_recovery_df.count()
)

distinct_payment_recovery_count = (
    payment_recovery_df
    .select("invoice_id")
    .distinct()
    .count()
)

duplicate_payment_recovery_count = (
    payment_recovery_count
    - distinct_payment_recovery_count
)

payment_recovery_summary = (
    payment_recovery_df
    .agg(
        F.sum(
            "first_attempt_success_flag"
        ).alias(
            "first_attempt_success_count"
        ),
        F.sum(
            "recovery_eligible_flag"
        ).alias(
            "recovery_eligible_count"
        ),
        F.sum(
            "recovery_success_flag"
        ).alias(
            "recovery_success_count"
        ),
        F.sum(
            "unrecovered_failure_flag"
        ).alias(
            "unrecovered_failure_count"
        ),
        F.sum(
            "pending_collection_flag"
        ).alias(
            "pending_collection_count"
        ),
        F.round(
            F.sum("recovered_amount"),
            2,
        ).alias(
            "recovered_amount"
        ),
        F.round(
            F.sum("unrecovered_amount"),
            2,
        ).alias(
            "unrecovered_amount"
        ),
        F.round(
            F.sum(
                "pending_collection_amount"
            ),
            2,
        ).alias(
            "pending_collection_amount"
        ),
    )
    .first()
)

recovery_rate_pct = round(
    (
        payment_recovery_summary[
            "recovery_success_count"
        ]
        / payment_recovery_summary[
            "recovery_eligible_count"
        ]
    ) * 100,
    2,
)


assert (
    payment_recovery_count
    == EXPECTED_PAYMENT_JOURNEY_COUNT
), (
    "Unexpected Payment Recovery record count."
)

assert duplicate_payment_recovery_count == 0, (
    "Duplicate Payment Recovery journeys detected."
)


print(
    "Payment Recovery records: "
    f"{payment_recovery_count:,}"
)

print(
    "Distinct invoice journeys: "
    f"{distinct_payment_recovery_count:,}"
)

print(
    "First-attempt successes: "
    f"{payment_recovery_summary['first_attempt_success_count']:,}"
)

print(
    "Recovery-eligible journeys: "
    f"{payment_recovery_summary['recovery_eligible_count']:,}"
)

print(
    "Successful recoveries: "
    f"{payment_recovery_summary['recovery_success_count']:,}"
)

print(
    "Unrecovered failures: "
    f"{payment_recovery_summary['unrecovered_failure_count']:,}"
)

print(
    "Pending collections: "
    f"{payment_recovery_summary['pending_collection_count']:,}"
)

print(
    "Retry recovery rate: "
    f"{recovery_rate_pct:,.2f}%"
)

print(
    "Recovered amount: "
    f"{payment_recovery_summary['recovered_amount']:,.2f}"
)

print(
    "Unrecovered amount: "
    f"{payment_recovery_summary['unrecovered_amount']:,.2f}"
)

print(
    "Pending collection amount: "
    f"{payment_recovery_summary['pending_collection_amount']:,.2f}"
)

display(
    payment_recovery_df
    .groupBy(
        "first_payment_provider",
        "first_payment_method",
        "recovery_outcome",
    )
    .agg(
        F.count("*").alias(
            "journey_count"
        ),
        F.round(
            F.sum("recovered_amount"),
            2,
        ).alias(
            "recovered_amount"
        ),
        F.round(
            F.sum("unrecovered_amount"),
            2,
        ).alias(
            "unrecovered_amount"
        ),
    )
    .orderBy(
        "first_payment_provider",
        "first_payment_method",
        "recovery_outcome",
    )
)

## 4. Validate and Reconcile Payment Recovery

Validate payment journey uniqueness, attempt sequencing, recovery outcomes, status flags, financial amounts, recovery duration, priority classification, and recommended actions.

Payment-attempt, settlement, retry recovery, unresolved exposure, and pending collection totals must reconcile exactly with the validated Silver payments and Revenue Leakage Gold source.

In [0]:
gold_hash_columns = [
    column_name
    for column_name in payment_recovery_df.columns
    if column_name not in {
        "_gold_generated_at",
        "_gold_record_hash",
    }
]

payment_recovery_df = (
    payment_recovery_df
    .withColumn(
        "_gold_record_hash",
        F.sha2(
            F.to_json(
                F.struct(
                    *[
                        F.col(column_name)
                        for column_name
                        in gold_hash_columns
                    ]
                ),
                options={
                    "ignoreNullFields": "false"
                },
            ),
            256,
        ),
    )
)


critical_fields = [
    "invoice_id",
    "customer_id",
    "subscription_id",
    "first_payment_status",
    "latest_payment_status",
    "recovery_outcome",
    "recovery_status",
    "recovery_priority",
    "recommended_recovery_action",
    "_gold_record_hash",
]

null_critical_field_condition = F.lit(False)

for column_name in critical_fields:
    null_critical_field_condition = (
        null_critical_field_condition
        | F.col(column_name).isNull()
    )


invalid_payment_count_condition = (
    F.col("payment_attempt_count")
    != (
        F.col("successful_payment_count")
        + F.col("failed_payment_count")
        + F.col("pending_payment_count")
    )
)

invalid_retry_count_condition = (
    F.col("retry_attempt_count")
    != (
        F.col("successful_retry_count")
        + F.col("failed_retry_count")
    )
)

invalid_attempt_sequence_condition = (
    (F.col("first_attempt_number") != 1)
    |
    (
        F.col("latest_attempt_number")
        != F.col("maximum_attempt_number")
    )
    |
    (
        F.col("payment_attempt_count")
        != F.col("distinct_payment_count")
    )
    |
    (
        F.col("payment_attempt_count")
        != F.col(
            "distinct_provider_transaction_count"
        )
    )
)

valid_recovery_outcome_condition = (
    (
        (
            F.col("first_payment_status")
            == "Succeeded"
        )
        & (
            F.col("latest_payment_status")
            == "Succeeded"
        )
        & (
            F.col("latest_attempt_number") == 1
        )
        & (
            F.col("recovery_outcome")
            == "First Attempt Success"
        )
        & (
            F.col("recovery_status")
            == "Resolved"
        )
    )
    |
    (
        (
            F.col("first_payment_status")
            == "Failed"
        )
        & (
            F.col("latest_payment_status")
            == "Succeeded"
        )
        & (
            F.col("latest_attempt_number") > 1
        )
        & (
            F.col("recovery_outcome")
            == "Recovered by Retry"
        )
        & (
            F.col("recovery_status")
            == "Recovered"
        )
    )
    |
    (
        (
            F.col("first_payment_status")
            == "Failed"
        )
        & (
            F.col("latest_payment_status")
            == "Failed"
        )
        & (
            F.col("latest_attempt_number") > 1
        )
        & (
            F.col("recovery_outcome")
            == "Retry Failed"
        )
        & (
            F.col("recovery_status")
            == "Unrecovered"
        )
    )
    |
    (
        (
            F.col("first_payment_status")
            == "Failed"
        )
        & (
            F.col("latest_payment_status")
            == "Failed"
        )
        & (
            F.col("latest_attempt_number") == 1
        )
        & (
            F.col("recovery_outcome")
            == "No Retry Attempted"
        )
        & (
            F.col("recovery_status")
            == "Unrecovered"
        )
    )
    |
    (
        (
            F.col("first_payment_status")
            == "Pending"
        )
        & (
            F.col("latest_payment_status")
            == "Pending"
        )
        & (
            F.col("recovery_outcome")
            == "Pending Collection"
        )
        & (
            F.col("recovery_status")
            == "Pending"
        )
    )
)

invalid_recovery_outcome_condition = (
    ~valid_recovery_outcome_condition
)


invalid_flag_condition = (
    (
        F.col("first_attempt_success_flag")
        != F.when(
            F.col("first_payment_status")
            == "Succeeded",
            1,
        ).otherwise(0)
    )
    |
    (
        F.col("recovery_eligible_flag")
        != F.when(
            F.col("first_payment_status")
            == "Failed",
            1,
        ).otherwise(0)
    )
    |
    (
        F.col("recovery_success_flag")
        != F.when(
            F.col("recovery_status")
            == "Recovered",
            1,
        ).otherwise(0)
    )
    |
    (
        F.col("unrecovered_failure_flag")
        != F.when(
            F.col("recovery_status")
            == "Unrecovered",
            1,
        ).otherwise(0)
    )
    |
    (
        F.col("pending_collection_flag")
        != F.when(
            F.col("recovery_status")
            == "Pending",
            1,
        ).otherwise(0)
    )
)


invalid_amount_condition = (
    (
        F.abs(
            F.col("settled_amount")
            - F.col("amount_paid")
        ) > 0.01
    )
    |
    (
        F.abs(
            F.col("recovered_amount")
            - F.col("recovered_retry_amount")
        ) > 0.01
    )
    |
    (
        F.col("recovered_amount")
        > F.col("settled_amount")
    )
    |
    (
        (
            F.col("recovery_status")
            == "Unrecovered"
        )
        & (
            F.abs(
                F.col("unrecovered_amount")
                - F.col("outstanding_amount")
            ) > 0.01
        )
    )
    |
    (
        (
            F.col("recovery_status")
            != "Unrecovered"
        )
        & (
            F.abs(
                F.col("unrecovered_amount")
            ) > 0.01
        )
    )
    |
    (
        (
            F.col("recovery_status")
            == "Pending"
        )
        & (
            F.abs(
                F.col(
                    "pending_collection_amount"
                )
                - F.col("pending_attempt_amount")
            ) > 0.01
        )
    )
    |
    (
        (
            F.col("recovery_status")
            != "Pending"
        )
        & (
            F.abs(
                F.col(
                    "pending_collection_amount"
                )
            ) > 0.01
        )
    )
)


invalid_recovery_duration_condition = (
    (
        (
            F.col("recovery_status")
            == "Recovered"
        )
        & (
            F.col(
                "recovery_duration_days"
            ).isNull()
            |
            (
                F.col(
                    "recovery_duration_days"
                ) < 0
            )
        )
    )
    |
    (
        (
            F.col("recovery_status")
            != "Recovered"
        )
        & (
            F.col(
                "recovery_duration_days"
            ).isNotNull()
        )
    )
)


invalid_priority_score_condition = (
    (
        F.col("recovery_priority_score")
        < 0
    )
    |
    (
        F.col("recovery_priority_score")
        > 100
    )
)

valid_priority_mapping_condition = (
    (
        (
            F.col("recovery_priority_score")
            >= 75
        )
        & (
            F.col("recovery_priority")
            == "Critical"
        )
    )
    |
    (
        (
            F.col("recovery_priority_score")
            .between(50, 74)
        )
        & (
            F.col("recovery_priority")
            == "High"
        )
    )
    |
    (
        (
            F.col("recovery_priority_score")
            .between(25, 49)
        )
        & (
            F.col("recovery_priority")
            == "Medium"
        )
    )
    |
    (
        (
            F.col("recovery_priority_score")
            .between(1, 24)
        )
        & (
            F.col("recovery_priority")
            == "Low"
        )
    )
    |
    (
        (
            F.col("recovery_priority_score")
            == 0
        )
        & (
            F.col("recovery_priority")
            == "None"
        )
    )
)

invalid_priority_mapping_condition = (
    ~valid_priority_mapping_condition
)


valid_action_mapping_condition = (
    (
        (
            F.col("recovery_outcome")
            == "Retry Failed"
        )
        & (
            F.col(
                "recommended_recovery_action"
            )
            == "Escalate Collection"
        )
    )
    |
    (
        (
            F.col("recovery_outcome")
            == "No Retry Attempted"
        )
        & (
            F.col(
                "recommended_recovery_action"
            )
            == "Initiate Retry"
        )
    )
    |
    (
        (
            F.col("recovery_outcome")
            == "Pending Collection"
        )
        & (
            F.col(
                "recommended_recovery_action"
            )
            == "Monitor Pending Payment"
        )
    )
    |
    (
        (
            F.col("recovery_outcome")
            == "Recovered by Retry"
        )
        & (
            F.col(
                "recommended_recovery_action"
            )
            == "Monitor Recovered Customer"
        )
    )
    |
    (
        (
            F.col("recovery_outcome")
            == "First Attempt Success"
        )
        & (
            F.col(
                "recommended_recovery_action"
            )
            == "No Action Required"
        )
    )
)

invalid_action_mapping_condition = (
    ~valid_action_mapping_condition
)


nonnegative_metric_columns = [
    "payment_attempt_count",
    "distinct_payment_count",
    "distinct_provider_transaction_count",
    "maximum_attempt_number",
    "successful_payment_count",
    "failed_payment_count",
    "pending_payment_count",
    "retry_attempt_count",
    "successful_retry_count",
    "failed_retry_count",
    "payment_attempt_amount",
    "settled_amount",
    "failed_attempt_amount",
    "pending_attempt_amount",
    "recovered_retry_amount",
    "first_attempt_amount",
    "first_settled_amount",
    "latest_attempt_amount",
    "latest_settled_amount",
    "recoverable_amount",
    "recovered_amount",
    "unrecovered_amount",
    "pending_collection_amount",
    "recovery_priority_score",
]

invalid_negative_metric_condition = (
    F.lit(False)
)

for column_name in nonnegative_metric_columns:
    invalid_negative_metric_condition = (
        invalid_negative_metric_condition
        | (F.col(column_name) < 0)
    )


validation_conditions = {
    "null_critical_field_count":
        null_critical_field_condition,
    "invalid_payment_count":
        invalid_payment_count_condition,
    "invalid_retry_count":
        invalid_retry_count_condition,
    "invalid_attempt_sequence_count":
        invalid_attempt_sequence_condition,
    "invalid_recovery_outcome_count":
        invalid_recovery_outcome_condition,
    "invalid_flag_count":
        invalid_flag_condition,
    "invalid_amount_count":
        invalid_amount_condition,
    "invalid_recovery_duration_count":
        invalid_recovery_duration_condition,
    "invalid_priority_score_count":
        invalid_priority_score_condition,
    "invalid_priority_mapping_count":
        invalid_priority_mapping_condition,
    "invalid_action_mapping_count":
        invalid_action_mapping_condition,
    "invalid_negative_metric_count":
        invalid_negative_metric_condition,
}


payment_recovery_validation_row = (
    payment_recovery_df
    .agg(
        F.count("*").alias(
            "payment_recovery_count"
        ),
        F.countDistinct("invoice_id").alias(
            "distinct_invoice_count"
        ),
        *[
            F.sum(
                F.when(
                    condition,
                    1,
                ).otherwise(0)
            ).alias(metric_name)
            for metric_name, condition
            in validation_conditions.items()
        ],
    )
    .first()
)

duplicate_invoice_count = (
    payment_recovery_validation_row[
        "payment_recovery_count"
    ]
    - payment_recovery_validation_row[
        "distinct_invoice_count"
    ]
)


silver_payment_totals = (
    silver_payments_df
    .agg(
        F.count("*").alias(
            "payment_attempt_count"
        ),
        F.round(
            F.sum("transaction_amount"),
            2,
        ).alias(
            "payment_attempt_amount"
        ),
        F.round(
            F.sum("settled_amount"),
            2,
        ).alias(
            "settled_amount"
        ),
        F.round(
            F.sum(
                F.when(
                    (
                        F.col("payment_status")
                        == "Succeeded"
                    )
                    & (
                        F.col("attempt_number") > 1
                    ),
                    F.col("settled_amount"),
                ).otherwise(0)
            ),
            2,
        ).alias(
            "recovered_amount"
        ),
        F.round(
            F.sum(
                F.when(
                    F.col("payment_status")
                    == "Pending",
                    F.col("transaction_amount"),
                ).otherwise(0)
            ),
            2,
        ).alias(
            "pending_amount"
        ),
    )
    .first()
)

gold_payment_totals = (
    payment_recovery_df
    .agg(
        F.sum("payment_attempt_count").alias(
            "payment_attempt_count"
        ),
        F.round(
            F.sum("payment_attempt_amount"),
            2,
        ).alias(
            "payment_attempt_amount"
        ),
        F.round(
            F.sum("settled_amount"),
            2,
        ).alias(
            "settled_amount"
        ),
        F.round(
            F.sum("recovered_amount"),
            2,
        ).alias(
            "recovered_amount"
        ),
        F.round(
            F.sum(
                "pending_collection_amount"
            ),
            2,
        ).alias(
            "pending_amount"
        ),
        F.sum(
            "recovery_eligible_flag"
        ).alias(
            "recovery_eligible_count"
        ),
        F.sum(
            "recovery_success_flag"
        ).alias(
            "recovery_success_count"
        ),
        F.round(
            F.sum("unrecovered_amount"),
            2,
        ).alias(
            "unrecovered_amount"
        ),
    )
    .first()
)

silver_recovery_eligible_count = (
    silver_payments_df
    .filter(
        (F.col("attempt_number") == 1)
        & (
            F.col("payment_status")
            == "Failed"
        )
    )
    .select("invoice_id")
    .distinct()
    .count()
)

silver_recovery_success_count = (
    silver_payments_df
    .filter(
        (F.col("attempt_number") > 1)
        & (
            F.col("payment_status")
            == "Succeeded"
        )
    )
    .select("invoice_id")
    .distinct()
    .count()
)

revenue_leakage_unrecovered_total = (
    revenue_leakage_source_df
    .filter(
        F.col("leakage_status")
        == "Confirmed"
    )
    .agg(
        F.round(
            F.sum(
                "confirmed_leakage_amount"
            ),
            2,
        ).alias(
            "unrecovered_amount"
        )
    )
    .first()["unrecovered_amount"]
)


reconciliation_pairs = {
    "payment_attempt_count": (
        silver_payment_totals[
            "payment_attempt_count"
        ],
        gold_payment_totals[
            "payment_attempt_count"
        ],
    ),
    "payment_attempt_amount": (
        silver_payment_totals[
            "payment_attempt_amount"
        ],
        gold_payment_totals[
            "payment_attempt_amount"
        ],
    ),
    "settled_amount": (
        silver_payment_totals[
            "settled_amount"
        ],
        gold_payment_totals[
            "settled_amount"
        ],
    ),
    "recovered_amount": (
        silver_payment_totals[
            "recovered_amount"
        ],
        gold_payment_totals[
            "recovered_amount"
        ],
    ),
    "pending_amount": (
        silver_payment_totals[
            "pending_amount"
        ],
        gold_payment_totals[
            "pending_amount"
        ],
    ),
    "recovery_eligible_count": (
        silver_recovery_eligible_count,
        gold_payment_totals[
            "recovery_eligible_count"
        ],
    ),
    "recovery_success_count": (
        silver_recovery_success_count,
        gold_payment_totals[
            "recovery_success_count"
        ],
    ),
    "unrecovered_amount": (
        revenue_leakage_unrecovered_total,
        gold_payment_totals[
            "unrecovered_amount"
        ],
    ),
}


assert (
    payment_recovery_validation_row[
        "payment_recovery_count"
    ]
    == EXPECTED_PAYMENT_JOURNEY_COUNT
), (
    "Unexpected Payment Recovery count."
)

assert duplicate_invoice_count == 0, (
    "Duplicate Payment Recovery invoices detected."
)

for metric_name in validation_conditions:
    assert (
        payment_recovery_validation_row[
            metric_name
        ]
        == 0
    ), (
        f"Validation failed: {metric_name}"
    )

for metric_name, (
    source_value,
    gold_value,
) in reconciliation_pairs.items():
    difference = abs(
        float(source_value or 0)
        - float(gold_value or 0)
    )

    assert difference <= 0.01, (
        f"Reconciliation failed: {metric_name}"
    )


print(
    "payment_recovery_count: "
    f"{payment_recovery_validation_row['payment_recovery_count']:,}"
)

print(
    "distinct_invoice_count: "
    f"{payment_recovery_validation_row['distinct_invoice_count']:,}"
)

for metric_name in validation_conditions:
    print(
        f"{metric_name}: "
        f"{payment_recovery_validation_row[metric_name]:,}"
    )

print(
    "duplicate_invoice_count: "
    f"{duplicate_invoice_count:,}"
)

for metric_name, (
    source_value,
    gold_value,
) in reconciliation_pairs.items():
    difference = abs(
        float(source_value or 0)
        - float(gold_value or 0)
    )

    print(
        f"{metric_name}: "
        f"Source={float(source_value or 0):,.2f}, "
        f"Gold={float(gold_value or 0):,.2f}, "
        f"Difference={difference:,.2f}"
    )

display(
    payment_recovery_df
    .groupBy(
        "recovery_priority",
        "recovery_status",
        "recommended_recovery_action",
    )
    .agg(
        F.count("*").alias(
            "journey_count"
        ),
        F.round(
            F.sum("recovered_amount"),
            2,
        ).alias(
            "recovered_amount"
        ),
        F.round(
            F.sum("unrecovered_amount"),
            2,
        ).alias(
            "unrecovered_amount"
        ),
        F.round(
            F.sum(
                "pending_collection_amount"
            ),
            2,
        ).alias(
            "pending_collection_amount"
        ),
    )
    .orderBy(
        "recovery_priority",
        "recovery_status",
        "recommended_recovery_action",
    )
)

## 5. Persist the Payment Recovery Gold Table

Persist one analytics-ready payment journey per invoice in a managed Delta table.

The initial execution creates the Gold table. Subsequent executions use the deterministic journey record hash and Delta `MERGE` to insert new journeys, update changed recovery outcomes, and remove journeys no longer present in the validated source.

In [0]:
spark.sql(
    "CREATE SCHEMA IF NOT EXISTS "
    "workspace.revenue_leakage_gold"
)

payment_recovery_df.createOrReplaceTempView(
    "payment_recovery_gold_updates"
)

if spark.catalog.tableExists(
    GOLD_PAYMENT_RECOVERY_TABLE
):
    spark.sql(
        f"""
        MERGE INTO {GOLD_PAYMENT_RECOVERY_TABLE} AS target
        USING payment_recovery_gold_updates AS source
            ON target.invoice_id = source.invoice_id

        WHEN MATCHED
            AND target._gold_record_hash
                <> source._gold_record_hash
            THEN UPDATE SET *

        WHEN NOT MATCHED
            THEN INSERT *

        WHEN NOT MATCHED BY SOURCE
            THEN DELETE
        """
    )

    write_method = "Delta MERGE"

else:
    (
        payment_recovery_df
        .write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            "true",
        )
        .saveAsTable(
            GOLD_PAYMENT_RECOVERY_TABLE
        )
    )

    write_method = (
        "Initial Delta table creation"
    )


saved_payment_recovery_df = spark.table(
    GOLD_PAYMENT_RECOVERY_TABLE
)

saved_payment_recovery_count = (
    saved_payment_recovery_df.count()
)

saved_distinct_invoice_count = (
    saved_payment_recovery_df
    .select("invoice_id")
    .distinct()
    .count()
)

saved_duplicate_invoice_count = (
    saved_payment_recovery_count
    - saved_distinct_invoice_count
)


source_saved_reconciliation = (
    payment_recovery_df
    .agg(
        F.round(
            F.sum("recovered_amount"),
            2,
        ).alias(
            "source_recovered_amount"
        ),
        F.round(
            F.sum("unrecovered_amount"),
            2,
        ).alias(
            "source_unrecovered_amount"
        ),
        F.round(
            F.sum(
                "pending_collection_amount"
            ),
            2,
        ).alias(
            "source_pending_amount"
        ),
    )
    .crossJoin(
        saved_payment_recovery_df
        .agg(
            F.round(
                F.sum("recovered_amount"),
                2,
            ).alias(
                "saved_recovered_amount"
            ),
            F.round(
                F.sum("unrecovered_amount"),
                2,
            ).alias(
                "saved_unrecovered_amount"
            ),
            F.round(
                F.sum(
                    "pending_collection_amount"
                ),
                2,
            ).alias(
                "saved_pending_amount"
            ),
        )
    )
    .first()
)


recovered_difference = abs(
    float(
        source_saved_reconciliation[
            "source_recovered_amount"
        ]
        or 0
    )
    - float(
        source_saved_reconciliation[
            "saved_recovered_amount"
        ]
        or 0
    )
)

unrecovered_difference = abs(
    float(
        source_saved_reconciliation[
            "source_unrecovered_amount"
        ]
        or 0
    )
    - float(
        source_saved_reconciliation[
            "saved_unrecovered_amount"
        ]
        or 0
    )
)

pending_difference = abs(
    float(
        source_saved_reconciliation[
            "source_pending_amount"
        ]
        or 0
    )
    - float(
        source_saved_reconciliation[
            "saved_pending_amount"
        ]
        or 0
    )
)


assert (
    saved_payment_recovery_count
    == EXPECTED_PAYMENT_JOURNEY_COUNT
), (
    "Unexpected saved Payment Recovery count."
)

assert (
    saved_distinct_invoice_count
    == EXPECTED_PAYMENT_JOURNEY_COUNT
), (
    "Unexpected saved distinct invoice count."
)

assert saved_duplicate_invoice_count == 0, (
    "Duplicate saved Payment Recovery journeys detected."
)

assert recovered_difference <= 0.01, (
    "Saved recovered amount does not reconcile."
)

assert unrecovered_difference <= 0.01, (
    "Saved unrecovered amount does not reconcile."
)

assert pending_difference <= 0.01, (
    "Saved pending amount does not reconcile."
)


print(
    "Write method: "
    f"{write_method}"
)

print(
    "Gold table: "
    f"{GOLD_PAYMENT_RECOVERY_TABLE}"
)

print(
    "Saved Payment Recovery records: "
    f"{saved_payment_recovery_count:,}"
)

print(
    "Saved distinct invoice journeys: "
    f"{saved_distinct_invoice_count:,}"
)

print(
    "Saved recovered difference: "
    f"{recovered_difference:,.2f}"
)

print(
    "Saved unrecovered difference: "
    f"{unrecovered_difference:,.2f}"
)

print(
    "Saved pending difference: "
    f"{pending_difference:,.2f}"
)

display(
    saved_payment_recovery_df
    .groupBy(
        "recovery_status",
        "recovery_priority",
    )
    .agg(
        F.count("*").alias(
            "journey_count"
        ),
        F.round(
            F.sum("recovered_amount"),
            2,
        ).alias(
            "recovered_amount"
        ),
        F.round(
            F.sum("unrecovered_amount"),
            2,
        ).alias(
            "unrecovered_amount"
        ),
        F.round(
            F.sum(
                "pending_collection_amount"
            ),
            2,
        ).alias(
            "pending_collection_amount"
        ),
    )
    .orderBy(
        "recovery_status",
        "recovery_priority",
    )
)

## 6. Validate Idempotent Payment Recovery Reprocessing

Rerun the Payment Recovery Delta merge using the same validated journey dataset.

The unchanged rerun must preserve all payment journeys and recovery totals while producing zero inserts, updates, or deletes.

In [0]:
payment_recovery_before_rerun = (
    spark.table(
        GOLD_PAYMENT_RECOVERY_TABLE
    )
    .agg(
        F.count("*").alias(
            "row_count"
        ),
        F.round(
            F.sum("recovered_amount"),
            2,
        ).alias(
            "recovered_amount"
        ),
        F.round(
            F.sum("unrecovered_amount"),
            2,
        ).alias(
            "unrecovered_amount"
        ),
        F.round(
            F.sum(
                "pending_collection_amount"
            ),
            2,
        ).alias(
            "pending_amount"
        ),
    )
    .first()
)

payment_recovery_df.createOrReplaceTempView(
    "payment_recovery_gold_updates"
)

spark.sql(
    f"""
    MERGE INTO {GOLD_PAYMENT_RECOVERY_TABLE} AS target
    USING payment_recovery_gold_updates AS source
        ON target.invoice_id = source.invoice_id

    WHEN MATCHED
        AND target._gold_record_hash
            <> source._gold_record_hash
        THEN UPDATE SET *

    WHEN NOT MATCHED
        THEN INSERT *

    WHEN NOT MATCHED BY SOURCE
        THEN DELETE
    """
)

payment_recovery_after_rerun_df = spark.table(
    GOLD_PAYMENT_RECOVERY_TABLE
)

payment_recovery_after_rerun = (
    payment_recovery_after_rerun_df
    .agg(
        F.count("*").alias(
            "row_count"
        ),
        F.countDistinct("invoice_id").alias(
            "distinct_invoice_count"
        ),
        F.round(
            F.sum("recovered_amount"),
            2,
        ).alias(
            "recovered_amount"
        ),
        F.round(
            F.sum("unrecovered_amount"),
            2,
        ).alias(
            "unrecovered_amount"
        ),
        F.round(
            F.sum(
                "pending_collection_amount"
            ),
            2,
        ).alias(
            "pending_amount"
        ),
    )
    .first()
)

duplicate_journeys_after_rerun = (
    payment_recovery_after_rerun[
        "row_count"
    ]
    - payment_recovery_after_rerun[
        "distinct_invoice_count"
    ]
)

latest_payment_recovery_history_df = (
    spark.sql(
        f"""
        DESCRIBE HISTORY
        {GOLD_PAYMENT_RECOVERY_TABLE}
        """
    )
    .orderBy(
        F.desc("version")
    )
    .limit(1)
)

latest_history_row = (
    latest_payment_recovery_history_df
    .first()
)

latest_operation_metrics = (
    latest_history_row[
        "operationMetrics"
    ]
    or {}
)

rows_inserted_during_rerun = int(
    latest_operation_metrics.get(
        "numTargetRowsInserted",
        0,
    )
)

rows_updated_during_rerun = int(
    latest_operation_metrics.get(
        "numTargetRowsUpdated",
        0,
    )
)

rows_deleted_during_rerun = int(
    latest_operation_metrics.get(
        "numTargetRowsDeleted",
        0,
    )
)

recovered_difference_after_rerun = abs(
    float(
        payment_recovery_before_rerun[
            "recovered_amount"
        ]
        or 0
    )
    - float(
        payment_recovery_after_rerun[
            "recovered_amount"
        ]
        or 0
    )
)

unrecovered_difference_after_rerun = abs(
    float(
        payment_recovery_before_rerun[
            "unrecovered_amount"
        ]
        or 0
    )
    - float(
        payment_recovery_after_rerun[
            "unrecovered_amount"
        ]
        or 0
    )
)

pending_difference_after_rerun = abs(
    float(
        payment_recovery_before_rerun[
            "pending_amount"
        ]
        or 0
    )
    - float(
        payment_recovery_after_rerun[
            "pending_amount"
        ]
        or 0
    )
)

assert (
    payment_recovery_before_rerun[
        "row_count"
    ]
    == EXPECTED_PAYMENT_JOURNEY_COUNT
), (
    "Unexpected row count before rerun."
)

assert (
    payment_recovery_after_rerun[
        "row_count"
    ]
    == EXPECTED_PAYMENT_JOURNEY_COUNT
), (
    "Unexpected row count after rerun."
)

assert rows_inserted_during_rerun == 0, (
    "Payment Recovery rerun inserted unexpected rows."
)

assert rows_updated_during_rerun == 0, (
    "Payment Recovery rerun updated unexpected rows."
)

assert rows_deleted_during_rerun == 0, (
    "Payment Recovery rerun deleted unexpected rows."
)

assert duplicate_journeys_after_rerun == 0, (
    "Duplicate journeys detected after rerun."
)

assert recovered_difference_after_rerun <= 0.01, (
    "Recovered amount changed during rerun."
)

assert unrecovered_difference_after_rerun <= 0.01, (
    "Unrecovered amount changed during rerun."
)

assert pending_difference_after_rerun <= 0.01, (
    "Pending amount changed during rerun."
)

print(
    "Rows before rerun: "
    f"{payment_recovery_before_rerun['row_count']:,}"
)

print(
    "Rows after rerun: "
    f"{payment_recovery_after_rerun['row_count']:,}"
)

print(
    "Rows inserted during rerun: "
    f"{rows_inserted_during_rerun:,}"
)

print(
    "Rows updated during rerun: "
    f"{rows_updated_during_rerun:,}"
)

print(
    "Rows deleted during rerun: "
    f"{rows_deleted_during_rerun:,}"
)

print(
    "Duplicate journeys after rerun: "
    f"{duplicate_journeys_after_rerun:,}"
)

print(
    "Recovered difference after rerun: "
    f"{recovered_difference_after_rerun:,.2f}"
)

print(
    "Unrecovered difference after rerun: "
    f"{unrecovered_difference_after_rerun:,.2f}"
)

print(
    "Pending difference after rerun: "
    f"{pending_difference_after_rerun:,.2f}"
)

print(
    "Payment Recovery Gold processing is idempotent."
)

display(
    latest_payment_recovery_history_df
    .select(
        "version",
        "timestamp",
        "operation",
        "operationMetrics",
    )
)

## 7. Final Result

The Payment Recovery Gold model was created successfully as an invoice-level payment journey Delta table.

### Output

- **Gold table:** `workspace.revenue_leakage_gold.payment_recovery`
- **Payment journey records:** 25,748
- **Distinct invoice journeys:** 25,748
- **Payment attempts reconciled:** 29,972
- **First-attempt successes:** 18,631
- **Recovery-eligible journeys:** 6,923
- **Successful retry recoveries:** 3,282
- **Unrecovered payment failures:** 3,641
- **Pending collections:** 194
- **Retry recovery rate:** 47.41%
- **Recovered retry revenue:** 583,858.90 USD
- **Unrecovered revenue:** 646,033.53 USD
- **Pending collection amount:** 31,490.34 USD
- **Critical unrecovered journeys:** 3,291
- **High-priority unrecovered journeys:** 350

### Quality Guarantees

- One record per invoice payment journey
- Zero duplicate payment journeys
- Zero null critical fields
- Valid first and latest payment-attempt sequencing
- Valid recovery outcomes, status flags, priorities, and actions
- Exact Silver payment-attempt reconciliation
- Exact settlement, recovery, pending, and unrecovered reconciliation
- Deterministic Gold record hashes
- Idempotent Delta merge processing
- Zero inserts, updates, deletes, or financial changes during unchanged reprocessing